## MTSA EXAMPLE

### Some imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")

import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score

from examples.sample_data_utils import get_acoustic_sample_data, get_tabular_sample_data
from mtsa.losses.forward_loss import ForwardLoss
from mtsa.preprocessing.feature_extraction.non_parameterized.vec2mfcc import Array2Mfcc
from mtsa.losses.lagrange_loss import LagrangeLoss
from mtsa.losses.trace_exponential_matrix_loss import TraceExponentialMatrixLoss
from mtsa.models.hybrid.ganf import GANF
from mtsa.predictors.standard_predictor import StandardPredictor
from mtsa.trainers.lagrange_trainer import LagrangeTrainer
from mtsa.utils.data import get_dataloader
from mtsa.utils.plots import plot_roc_pr_curves
from mtsa.utils.utils import Wav2Array


## Building a model


### Ganf Model

#### Synthetic Data

> **Desafio 1 — divisão dos dados.** A função `get_tabular_sample_data` original injetava anomalias na série inteira e depois separava normais e anômalas com máscara + `np.vstack`. Isso **emendava** trechos distantes da série e **empilhava todas as anomalias no fim do teste**, destruindo a ordem temporal que o modelo usa (janelas de 12 instantes seguidos). Havia ainda um erro no intervalo `(start_point or step, step)`, que encurtava uma anomalia e deixava outra vazia.
>
> **Correção** (`examples/sample_data_utils.py`): dividir a série pelo tempo **antes** (70% iniciais = treino, 30% finais = teste) e injetar as anomalias **só no teste**, no próprio lugar. A versão da aula ficou intacta em `examples/sample_data_utils_original.py`.
>
> Para alternar, mude `USAR_VERSAO_CORRIGIDA` abaixo e rode esta seção de novo: as células seguintes usam a versão escolhida e o título do gráfico mostra qual foi.

In [ ]:
from examples import sample_data_utils, sample_data_utils_original

USAR_VERSAO_CORRIGIDA = True   # False = código original da aula

get_tabular_sample_data = (sample_data_utils if USAR_VERSAO_CORRIGIDA else sample_data_utils_original).get_tabular_sample_data
nome_versao = "CORRIGIDA" if USAR_VERSAO_CORRIGIDA else "ORIGINAL"
print("Usando a versão", nome_versao)

In [ ]:
X_train, y_train, X_test, y_test = get_tabular_sample_data()

training_samples = get_dataloader(
    X_train,
    y_train,
    window_size=12,
    stride_size=1,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
)

test_samples = get_dataloader(
    X_test,
    y_test,
    window_size=12,
    stride_size=1,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
)

> **Por que a versão original está errada.** O CSV está **em ordem** (`time` = 0, 1, …, 4999). A célula abaixo roda as duas funções **de verdade**, trocando a coluna X3 (que nunca recebe anomalia) pelo instante original de cada linha, para mostrar de que instante veio cada linha que o modelo recebe. Numa série íntegra, de uma linha para a seguinte o instante sempre aumenta 1; qualquer outro salto é uma **emenda**.
> - **Original:** 1 emenda no treino e 11 no teste. Numa delas o tempo **volta** de 1151 para 26, exatamente onde os rótulos mudam de 0 para 1; a janela de exemplo mostra o modelo recebendo esses instantes como se fossem consecutivos.
> - **Corrigida:** 0 emendas; a janela de exemplo tem instantes consecutivos, com a anomalia começando no meio de dados normais.
> - **Gráfico:** série íntegra = reta contínua; na original aparecem degraus, uma queda e as anomalias (vermelho) amontoadas no fim.

In [ ]:
from examples.experimentos_desafio1 import diagnostico_ordem_temporal

diagnostico_ordem_temporal()

In [ ]:
# Hypothesis set
model_ganf = GANF(
    n_blocks=6,
    input_features=1,
    hidden_size=32,
    n_hidden=1,
    model="MAF",
    device="cpu",
    time_series_amount=X_train.shape[1],
)

# Optimizer
weight_decay = 0.1
learning_rate = 1e-4
optimizer = torch.optim.Adam(
    [
        {"params": model_ganf.parameters(), "weight_decay": weight_decay},
        {"params": [model_ganf.adjacency_matrix]},
    ],
    lr=learning_rate,
    weight_decay=0.0,
)

# Error Measure
negative_log_prob_loss = ForwardLoss()
dag_loss = TraceExponentialMatrixLoss()
loss = LagrangeLoss(primal_loss=negative_log_prob_loss, dual_losses=[dag_loss])

In [ ]:
# Learning algorithm
learning_algorithm = LagrangeTrainer()

learning_algorithm.fit(    
    model=model_ganf,
    optimizer=optimizer,
    loss=loss,
    train_loader = training_samples,
    epochs=2,
    interactions=2
)

In [ ]:
predict_algorithm = StandardPredictor()

y_hat, y = predict_algorithm.predict(function=model_ganf, dataloader=test_samples, pointwise=True, aggregation="mean")



In [ ]:
roc_auc, pr_auc = plot_roc_pr_curves(y, y_hat, title=f"GANF | dados sintéticos | versão {nome_versao}")

##### Comparação dos cenários A, B, C e D

Duas coisas influenciam o resultado: **a divisão** (original ou corrigida) e **o tamanho das anomalias**. As originais (≈1,4 e ≈4,9) valem de 10 a 47 desvios-padrão das colunas, fáceis demais; as de **k desvios** são mais realistas.

| | Divisão | Anomalias | Para que serve |
|---|---|---|---|
| A | original | originais | referência: o resultado da aula |
| B | corrigida | originais | efeito só da correção |
| C | original | k desvios | quanto a divisão errada "ajuda" quando o problema é difícil |
| D | corrigida | k desvios | o resultado mais realista |

**Como ler:** A × B e C × D mostram o efeito da divisão; A × C e B × D, o do tamanho das anomalias. A e B usam os arquivos de verdade; C e D usam uma réplica de cada versão em que só o tamanho da anomalia muda. O GANF é treinado com a mesma configuração das células acima.

In [ ]:
from examples.experimentos_desafio1 import rodar_cenarios_ganf, tabela_resultados, graficos_comparativos, notas_ao_longo_do_tempo

K = 2  # tamanho das anomalias nos cenários C e D, em desvios-padrão

resultados = rodar_cenarios_ganf(k=K)   # treina os 4 cenários e mostra o gráfico de cada um
tabela_resultados(resultados)

Cor = tamanho da anomalia (laranja = original, azul = k desvios). Traço = divisão (tracejado = original, contínuo = corrigida). No gráfico de barras, o traço preto sobre cada barra de AP é o valor do acaso, que depende da % de anomalias no teste; por isso a AP só deve ser comparada com o próprio acaso.

In [ ]:
graficos_comparativos(resultados)

Nota do modelo em cada instante do teste (faixas vermelhas = anomalias do gabarito). Na divisão original, as anomalias aparecem todas juntas no fim; na corrigida, espalhadas no meio dos dados normais. Se a nota sobe antes e continua alta depois de uma faixa vermelha, os normais vizinhos estão herdando a nota da anomalia (efeito da média por janelas no preditor).

In [ ]:
notas_ao_longo_do_tempo(resultados)

#### Acoustic Data

In [ ]:
X_train, y_train, X_test, y_test = get_acoustic_sample_data()


preprocessing_steps = nn.Sequential(
    Wav2Array(sampling_rate=16000),
    Array2Mfcc(sampling_rate=16000),
)

X_train = preprocessing_steps(X_train)
X_test = preprocessing_steps(X_test)

X_train = torch.permute(X_train,(0,2,1))
X_test = torch.permute(X_test,(0,2,1))

training_samples = get_dataloader(
    X_train,
    y_train,
    window_size=12,
    stride_size=1,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
)

test_samples = get_dataloader(
    X_test,
    y_test,
    window_size=12,
    stride_size=1,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    persistent_workers=False,
    drop_last=False,
)

In [ ]:
X_train.shape

In [ ]:
# Hypothesis set
model_ganf = GANF(
    n_blocks=6,
    input_features=1,
    hidden_size=32,
    n_hidden=1,
    model="MAF",
    device="cpu",
    time_series_amount=X_train.shape[2],
)

# Optimizer
weight_decay = 0.1
learning_rate = 1e-4
optimizer = torch.optim.Adam(
    [
        {"params": model_ganf.parameters(), "weight_decay": weight_decay},
        {"params": [model_ganf.adjacency_matrix]},
    ],
    lr=learning_rate,
    weight_decay=0.0,
)

# Error Measure
negative_log_prob_loss = ForwardLoss()
dag_loss = TraceExponentialMatrixLoss()
loss = LagrangeLoss(primal_loss=negative_log_prob_loss, dual_losses=[dag_loss])

In [ ]:
# Learning algorithm
learning_algorithm = LagrangeTrainer()

learning_algorithm.fit(    
    model=model_ganf,
    optimizer=optimizer,
    loss=loss,
    train_loader = training_samples,
    epochs=2,
    interactions=2
)

In [ ]:
roc_auc, pr_auc = plot_roc_pr_curves(y, y_hat)